In [3]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import os

# Paths
X_path = "E:/LogAnomalyDetector/data/X_features.npz"
y_path = "E:/LogAnomalyDetector/data/y_labels.csv"
model_path = "E:/LogAnomalyDetector/models/best_model.joblib"
vec_path = "E:/LogAnomalyDetector/models/tfidf_vectorizer.joblib"


In [4]:
# Load feature matrix
X = sparse.load_npz(X_path)

# Load labels
y_df = pd.read_csv(y_path)
y = y_df["true_label"].astype(int)

# Load trained model
model = joblib.load(model_path)

# Load vectorizer (for future inference consistency)
vectorizer = joblib.load(vec_path)

print("Loaded X shape:", X.shape)
print("Loaded y shape:", y.shape)
print("Model type:", type(model))
print("Vectorizer loaded.")


Loaded X shape: (5, 51)
Loaded y shape: (5,)
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Vectorizer loaded.


In [5]:
# Predict on full dataset
y_pred = model.predict(X)

print("Confusion Matrix (Full Dataset):")
print(confusion_matrix(y, y_pred))

print("\nClassification Report (Full Dataset):")
print(classification_report(y, y_pred, zero_division=0))

acc = accuracy_score(y, y_pred)
print("\nOverall Accuracy:", acc)


Confusion Matrix (Full Dataset):
[[3 0]
 [0 2]]

Classification Report (Full Dataset):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         2

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5


Overall Accuracy: 1.0


In [6]:
# Save evaluation summary
results = {
    "accuracy": acc,
    "confusion_matrix": confusion_matrix(y, y_pred).tolist(),
    "classification_report": classification_report(y, y_pred, zero_division=0)
}

results_path = "E:/LogAnomalyDetector/reports/evaluation_results.json"
os.makedirs("E:/LogAnomalyDetector/reports", exist_ok=True)

import json
with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print("Evaluation results saved to:", results_path)


Evaluation results saved to: E:/LogAnomalyDetector/reports/evaluation_results.json


In [7]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os

# ==============================
# 1. LOAD DATA
# ==============================
X = sparse.load_npz("E:/LogAnomalyDetector/data/X_features.npz")
y = pd.read_csv("E:/LogAnomalyDetector/data/y_labels.csv")["true_label"].values

print("Dataset shape:", X.shape)

# ==============================
# 2. DATASET SIZE HANDLING
# ==============================
n_samples = len(y)

if n_samples < 20:
    print("⚠ Very small dataset → training on full data")
    X_train, X_test, y_train, y_test = X, X, y, y

elif n_samples < 100:
    print("⚠ Small dataset → using 80/20 split")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

else:
    print("✅ Normal dataset → stratified split")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

# ==============================
# 3. TRAIN MODEL
# ==============================
print("Training Random Forest...")

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# ==============================
# 4. PROBABILITY PREDICTION
# ==============================
y_prob = rf.predict_proba(X_test)[:, 1]

# 🔥 Adaptive threshold
if n_samples < 20:
    threshold = 0.5
elif n_samples < 100:
    threshold = 0.6
else:
    threshold = 0.7

print(f"Using threshold: {threshold}")

y_pred = (y_prob >= threshold).astype(int)

# ==============================
# 5. EVALUATION
# ==============================
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ==============================
# 6. FEATURE IMPORTANCE
# ==============================
importances = rf.feature_importances_

print("\nTop 10 Important Features:")
print(sorted(importances, reverse=True)[:10])

# ==============================
# 7. SAVE MODEL
# ==============================
os.makedirs("E:/LogAnomalyDetector/models", exist_ok=True)

joblib.dump(rf, "E:/LogAnomalyDetector/models/best_model.joblib")

print("✅ Model training complete & saved")

Dataset shape: (5, 51)
⚠ Very small dataset → training on full data
Training Random Forest...
Using threshold: 0.5

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         2

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5


Confusion Matrix:
[[3 0]
 [0 2]]

Top 10 Important Features:
[np.float64(0.058346560846560834), np.float64(0.0537037037037037), np.float64(0.04615319865319866), np.float64(0.04055555555555555), np.float64(0.037857142857142874), np.float64(0.03574074074074074), np.float64(0.031481481481481485), np.float64(0.031060606060606046), np.float64(0.030555555555555555), np.float64(0.0292881192881193)]
✅ Model training complete & saved


In [8]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[3 0]
 [0 2]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         2

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5

